# Standalone OOF Feature SHAP

保存済みの特徴量キャッシュからサンプルを作り、OOF Feature SHAPを計算します。

In [ ]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap

from sklearn.model_selection import GroupKFold
from lightgbm import LGBMClassifier, LGBMRegressor

ROOT = Path('C:/masterresearch/Comparative_advantage')
GAEZ_DIR = ROOT / 'GAEZ'
HYDE_DIR = ROOT / 'HYDE3.4'
CROPLAND_DIR = HYDE_DIR / 'cropland_npys'
DIST_DIR = ROOT / 'distance_to_cities'
GLOFAS_DIR = ROOT / 'GloFAS' / 'processed_5min'
FEATURE_CACHE = GAEZ_DIR / 'CroplandRegression' / 'features_cache'
OUTPUT_DIR = GAEZ_DIR / 'CroplandRegression' / 'standalone_oof_shap'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

RANDOM_SEED = 42
N_SPLITS = 5
YEAR = 2024
PRESENCE_THRESHOLD = 0.01
N_POS_SAMPLE = 120_000
N_ZERO_SAMPLE = 120_000
SHAP_MAX_ROWS_PER_FOLD = 10_000
SHAP_BACKGROUND_ROWS = 500

def safe_log1p(values):
    values = np.asarray(values, dtype=np.float32)
    values = np.where(np.isfinite(values) & (values > 0), values, 0.0)
    return np.log1p(values).astype(np.float32)

def load_cache(*names):
    for name in names:
        path = FEATURE_CACHE / name
        if path.exists():
            return np.load(path, mmap_mode='r')
    raise FileNotFoundError(f'Feature cache not found: {names}')

def choose_indices(indices, max_rows, random_state):
    indices = np.asarray(indices, dtype=int)
    if max_rows is None or len(indices) <= max_rows:
        return indices
    rng = np.random.default_rng(random_state)
    return np.sort(rng.choice(indices, max_rows, replace=False))

def shap_array(model, X, background, model_type):
    output_type = 'probability' if model_type == 'classifier' else 'raw'
    try:
        explainer = shap.TreeExplainer(model, data=background, model_output=output_type)
        values = explainer.shap_values(X, check_additivity=False)
    except Exception as error:
        print('Probability-scale SHAP failed; using raw output:', repr(error))
        values = shap.TreeExplainer(model).shap_values(X, check_additivity=False)
    if isinstance(values, list):
        values = values[1] if len(values) == 2 else values[0]
    values = np.asarray(values)
    if values.ndim == 3:
        values = values[:, :, -1]
    if values.ndim != 2:
        raise ValueError(f'Unexpected SHAP shape: {values.shape}')
    return values.astype(np.float32)

def add_feature_shap(base, values, weights, fold, stage):
    out = base[['row', 'col', 'lat', 'lon', 'spatial_block']].copy()
    out['cv_fold'] = fold
    out['stage'] = stage
    out['shap_weight'] = weights
    for j, feature in enumerate(feature_cols):
        out[f'{feature}__shap_signed'] = values[:, j]
        out[f'{feature}__shap_abs'] = np.abs(values[:, j])
    return out

def weighted_mean(values, weights):
    values = np.asarray(values, dtype=float)
    weights = np.asarray(weights, dtype=float)
    valid = np.isfinite(values) & np.isfinite(weights) & (weights > 0)
    if not valid.any():
        return np.nan
    return float(np.average(values[valid], weights=weights[valid]))


In [ ]:
# ============================================================
# データ読み込み
# ============================================================

lat = np.load(CROPLAND_DIR / 'lat.npy')
lon = np.load(CROPLAND_DIR / 'lon.npy')
cropland_raw = np.load(CROPLAND_DIR / 'cropland_fraction_1950_2024.npy', mmap_mode='r')
years = np.load(CROPLAND_DIR / 'years.npy')
year_index = int(np.where(years == YEAR)[0][0])
cropland_2024 = np.array(cropland_raw[year_index], dtype=np.float32, copy=True)
cropland_2024[~np.isfinite(cropland_2024)] = np.nan

population = load_cache('population_density_2024.npy')
elevation = load_cache('elevation_5min.npy')
slope = load_cache('slope_5min.npy')
exclusion = load_cache('exclusion_5min_mode.npy')

city_time = np.load(DIST_DIR / 'cities_10_1_12deg_min.npy', mmap_mode='r')
port_time = np.load(DIST_DIR / 'ports_05_1_12deg_min.npy', mmap_mode='r')
glofas = np.load(GLOFAS_DIR / 'p10_discharge_max_5min_2020.npy', mmap_mode='r')
distance_river = np.load(GLOFAS_DIR / 'distance_to_reliable_river_p10_gt_10_m3s_km_5min_2020.npy', mmap_mode='r')

rainfed_value = load_cache('rainfed_value_top5_usd_per_ha_checked_36crops.npy', 'rainfed_value_top5_checked_36crops.npy')
irrigated_value = load_cache('irrigated_value_top5_usd_per_ha_checked_36crops.npy', 'irrigated_value_top5_checked_36crops.npy')
rainfed_calorie = load_cache('rainfed_calorie_top5_kcal_per_ha_checked_36crops.npy', 'rainfed_calorie_top5_checked_36crops.npy')
irrigated_calorie = load_cache('irrigated_calorie_top5_kcal_per_ha_checked_36crops.npy', 'irrigated_calorie_top5_checked_36crops.npy')

print('rasters and cached features loaded')


In [ ]:
# ============================================================
# 学習用サンプルとSpatial GroupKFoldの準備
# ============================================================

land_mask = (
    np.isfinite(elevation)
    & np.isfinite(cropland_2024)
    & np.isfinite(population)
    & (population >= 0)
)
presence = land_mask & (cropland_2024 > PRESENCE_THRESHOLD)
absence = land_mask & ~presence

rng = np.random.default_rng(RANDOM_SEED)
positive_flat = np.flatnonzero(presence.ravel())
zero_flat = np.flatnonzero(absence.ravel())
positive_sample = rng.choice(positive_flat, min(N_POS_SAMPLE, len(positive_flat)), replace=False)
zero_sample = rng.choice(zero_flat, min(N_ZERO_SAMPLE, len(zero_flat)), replace=False)
sample_flat = np.concatenate([positive_sample, zero_sample])
rng.shuffle(sample_flat)
rows, cols = np.unravel_index(sample_flat, cropland_2024.shape)

def take(values):
    return np.asarray(values[rows, cols])

rainfed_value_sample = take(rainfed_value)
irrigated_value_sample = take(irrigated_value)
rainfed_calorie_sample = take(rainfed_calorie)
irrigated_calorie_sample = take(irrigated_calorie)

sample = pd.DataFrame({
    'row': rows.astype(np.int32),
    'col': cols.astype(np.int32),
    'lat': lat[rows].astype(np.float32),
    'lon': lon[cols].astype(np.float32),
    'cropland_fraction': take(cropland_2024).astype(np.float32),
    'presence': (take(cropland_2024) > PRESENCE_THRESHOLD).astype(np.uint8),
    'elevation_m': take(elevation).astype(np.float32),
    'slope': take(slope).astype(np.float32),
    'exclusion_class': np.nan_to_num(take(exclusion), nan=-1).astype(np.int16),
    'log_pop_density_2024': safe_log1p(take(population)),
    'log_city_time_20k_min': safe_log1p(take(city_time)),
    'log_port_time_any_min': safe_log1p(take(port_time)),
    'log_glofas_p10_2020': safe_log1p(take(glofas)),
    'log_distance_river_gt10_2020': safe_log1p(take(distance_river)),
    'log_rainfed_value_top5': safe_log1p(rainfed_value_sample),
    'log_rainfed_calorie_top5': safe_log1p(rainfed_calorie_sample),
    'log_irrigation_value_gain_top5': safe_log1p(np.maximum(irrigated_value_sample - rainfed_value_sample, 0)),
    'log_irrigation_calorie_gain_top5': safe_log1p(np.maximum(irrigated_calorie_sample - rainfed_calorie_sample, 0)),
})

sample = sample.replace([np.inf, -np.inf], np.nan).dropna().reset_index(drop=True)
sample['spatial_block'] = (
    np.floor((sample['lat'] + 90) / 10).astype(int) * 36
    + np.floor((sample['lon'] + 180) / 10).astype(int)
)

feature_cols = [
    'elevation_m', 'slope', 'exclusion_class',
    'log_pop_density_2024', 'log_city_time_20k_min',
    'log_port_time_any_min', 'log_glofas_p10_2020',
    'log_distance_river_gt10_2020', 'log_rainfed_value_top5',
    'log_rainfed_calorie_top5', 'log_irrigation_value_gain_top5',
    'log_irrigation_calorie_gain_top5',
]

cv_sample = sample.reset_index(drop=True)
y_all = cv_sample['presence'].to_numpy(dtype=np.uint8)
groups = cv_sample['spatial_block'].to_numpy()
gkf = GroupKFold(n_splits=N_SPLITS)

cell_area_km2 = np.load(CROPLAND_DIR / 'grid_area_km2.npy', mmap_mode='r')
sample_area_km2 = np.asarray(
    cell_area_km2[
        cv_sample['row'].to_numpy(dtype=int),
        cv_sample['col'].to_numpy(dtype=int),
    ],
    dtype=float,
)
n_pos_population = int(presence.sum())
n_zero_population = int(land_mask.sum() - n_pos_population)
n_pos_sample = int((y_all == 1).sum())
n_zero_sample = int((y_all == 0).sum())
area_population_weight = np.where(
    y_all == 1,
    n_pos_population / n_pos_sample,
    n_zero_population / n_zero_sample,
) * sample_area_km2

print('sample rows:', len(cv_sample))
print('sample presence share:', y_all.mean())


In [ ]:
# ============================================================
# OOF Feature SHAPの計算・保存・可視化
# ============================================================

oof_rows = []

for fold, (train_idx, test_idx) in enumerate(
    gkf.split(cv_sample, y_all, groups=groups),
    start=1,
):
    print(f'OOF FEATURE SHAP FOLD {fold}/{N_SPLITS}')

    train = cv_sample.iloc[train_idx].copy()
    test_idx_shap = choose_indices(test_idx, SHAP_MAX_ROWS_PER_FOLD, RANDOM_SEED + fold * 10)
    test = cv_sample.iloc[test_idx_shap].copy()

    clf = LGBMClassifier(
        objective='binary', n_estimators=450, learning_rate=0.035,
        num_leaves=31, min_child_samples=80, subsample=0.85,
        colsample_bytree=0.85, random_state=RANDOM_SEED + fold,
        n_jobs=4, verbose=-1,
    )
    clf.fit(train[feature_cols], train['presence'], categorical_feature=['exclusion_class'])

    background = train[feature_cols].sample(
        n=min(SHAP_BACKGROUND_ROWS, len(train)),
        random_state=RANDOM_SEED + fold,
    )
    stage1_values = shap_array(clf, test[feature_cols], background, 'classifier')
    oof_rows.append(add_feature_shap(
        test, stage1_values, area_population_weight[test_idx_shap], fold, 'presence'
    ))

    positive_train = train[train['presence'].eq(1)].copy()
    positive_test_idx = test_idx[y_all[test_idx] == 1]

    if len(positive_train) and len(positive_test_idx):
        reg = LGBMRegressor(
            objective='regression', n_estimators=550, learning_rate=0.03,
            num_leaves=31, min_child_samples=60, subsample=0.85,
            colsample_bytree=0.85, random_state=RANDOM_SEED + fold,
            n_jobs=4, verbose=-1,
        )
        reg.fit(
            positive_train[feature_cols],
            positive_train['cropland_fraction'],
            categorical_feature=['exclusion_class'],
        )

        stage2_idx = choose_indices(
            positive_test_idx, SHAP_MAX_ROWS_PER_FOLD, RANDOM_SEED + fold * 100
        )
        stage2_test = cv_sample.iloc[stage2_idx].copy()
        background_positive = positive_train[feature_cols].sample(
            n=min(SHAP_BACKGROUND_ROWS, len(positive_train)),
            random_state=RANDOM_SEED + fold,
        )
        stage2_values = shap_array(
            reg, stage2_test[feature_cols], background_positive, 'regressor'
        )
        oof_rows.append(add_feature_shap(
            stage2_test, stage2_values, area_population_weight[stage2_idx],
            fold, 'conditional_fraction'
        ))

    print('  stage 1 rows:', len(test_idx_shap))
    print('  stage 2 rows:', len(positive_test_idx))
    gc.collect()

oof_feature_shap = pd.concat(oof_rows, ignore_index=True)
values_path = OUTPUT_DIR / 'oof_feature_shap_values.csv'
oof_feature_shap.to_csv(values_path, index=False, encoding='utf-8-sig')

summary_rows = []
for stage, stage_df in oof_feature_shap.groupby('stage'):
    for feature in feature_cols:
        summary_rows.append({
            'stage': stage,
            'feature': feature,
            'mean_absolute_shap': weighted_mean(stage_df[f'{feature}__shap_abs'], stage_df['shap_weight']),
            'mean_signed_shap': weighted_mean(stage_df[f'{feature}__shap_signed'], stage_df['shap_weight']),
        })

summary = pd.DataFrame(summary_rows)
summary['relative_importance_share'] = (
    summary.groupby('stage')['mean_absolute_shap']
    .transform(lambda values: values / values.sum())
)
summary = summary.sort_values(['stage', 'mean_absolute_shap'], ascending=[True, False])
summary_path = OUTPUT_DIR / 'oof_feature_shap_summary.csv'
summary.to_csv(summary_path, index=False, encoding='utf-8-sig')
display(summary.round(5))

plot_df = summary.pivot(index='feature', columns='stage', values='mean_absolute_shap').fillna(0)
plot_df['total'] = plot_df.sum(axis=1)
plot_df = plot_df.sort_values('total', ascending=False).drop(columns='total')
ax = plot_df.plot(kind='bar', figsize=(15, 6))
ax.set_ylabel('Weighted mean absolute SHAP')
ax.set_xlabel('Feature')
ax.set_title('OOF SHAP importance by individual feature')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
figure_path = OUTPUT_DIR / 'oof_feature_shap_importance.png'
plt.savefig(figure_path, dpi=250, bbox_inches='tight')
plt.show()

print('Saved:', values_path)
print('Saved:', summary_path)
print('Saved:', figure_path)
